# RecipeGPT Replication with GPT-2 Small

This notebook fine-tunes **GPT-2 Small** to generate cooking instructions from a recipe title and ingredient list.

## Core experiments

1. Prepare a 20,000 / 2,000 / 2,000 train-validation-test split.
2. Run a small pilot experiment before the final training run.
3. Fine-tune GPT-2 Small with checkpoint saving to Google Drive.
4. Compare pretrained and fine-tuned GPT-2.
5. Compare greedy, beam, top-k, top-p, and temperature decoding.
6. Evaluate outputs using BLEU, ROUGE-L, BERTScore, Distinct-1, and Distinct-2.
7. Stretch goal: run a LoRA experiment.

> **Before running:** In Colab, select **Runtime → Change runtime type → T4 GPU**.

## 1. Install dependencies

In [1]:
!pip -q install -U transformers datasets accelerate evaluate     rouge-score sacrebleu bert-score peft

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.4 MB/s eta 0:00:00


## 2. Imports and reproducibility

In [2]:
import ast
import gc
import glob
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Mount Google Drive

The notebook saves dataset splits, checkpoints, models, and generated outputs to Drive. This protects the project if the Colab session disconnects.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/Shareddrives/RecipeGPT")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models" / "gpt2_full"
RESULTS_DIR = PROJECT_DIR / "results"

for folder in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Mounted at /content/drive
Project directory: /content/drive/Shareddrives/RecipeGPT


## 4. Configuration

Start with `PILOT_MODE = True`. After the pilot completes successfully, change it to `False` for the final experiment. When `PILOT_MODE = FALSE`, the `if PILOT_MODE` block is skipped, so it uses `Train: 20000, Validation: 2000, Test: 2000`.

Place the RecipeNLG file at:

`MyDrive/recipe_gpt_project/data/full_dataset.csv`

The original RecipeNLG CSV normally contains columns such as `title`, `ingredients`, and `directions`.

In [4]:
MODEL_NAME = "gpt2"

PILOT_MODE = False   # True: 1,000 training examples, 200 validation, 200 test
                    # False: 20,000 training, 2,000 validation, 2,000 test

RUN_SUFFIX = "pilot" if PILOT_MODE else "final"

if PILOT_MODE:
    NUM_EPOCHS = 1
    OUTPUT_DIR = MODEL_DIR / "pilot"
else:
    NUM_EPOCHS = 2
    OUTPUT_DIR = MODEL_DIR / "final_run"

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5

GENERATION_SAMPLE_SIZE = 50 if PILOT_MODE else 500
# Reason choosing 500: a good balance between: getting reliable evaluation metrics,
# keeping generation time reasonable,
# and reducing the amount of computation for BERTScore (which is the slowest metric).

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "pilot_mode": PILOT_MODE,
    "epochs": NUM_EPOCHS,
    "output_dir": str(OUTPUT_DIR),
})

{'pilot_mode': False, 'epochs': 2, 'output_dir': '/content/drive/Shareddrives/RecipeGPT/models/gpt2_full/final_run'}


## 5. Load and inspect prepared splits (RecipeNLG)

In [5]:
TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. "
            "Run the RecipeNLG Dataset Preparation notebook first."
        )

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Full prepared datasets:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Full prepared datasets:
Train: 20000
Validation: 2000
Test: 2000


In [6]:
if PILOT_MODE:
    train_df = train_df.sample(
        n=1000,
        random_state=42
    ).reset_index(drop=True)

    val_df = val_df.sample(
        n=200,
        random_state=42
    ).reset_index(drop=True)

    test_df = test_df.sample(
        n=200,
        random_state=42
    ).reset_index(drop=True)

print("Datasets used in this run:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Datasets used in this run:
Train: 20000
Validation: 2000
Test: 2000


## 6. Column Validation

In [7]:
REQUIRED_COLUMNS = ["title", "ingredients", "directions"]

for split_name, dataframe in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():
    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{split_name} is missing columns: {missing_columns}"
        )

    dataframe.dropna(
        subset=REQUIRED_COLUMNS,
        inplace=True,
    )

    for column in REQUIRED_COLUMNS:
        dataframe[column] = (
            dataframe[column]
            .astype(str)
            .str.strip()
        )

    dataframe.reset_index(drop=True, inplace=True)

print("Dataset columns validated and cleaned.")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Dataset columns validated and cleaned.
Train: (20000, 3)
Validation: (2000, 3)
Test: (2000, 3)


## 7. Format recipes for causal language modeling

The model sees a single sequence:

```text
Title: ...
Ingredients: ...
Instructions: ...
```

During generation, the prompt ends after `Instructions:`.

In [15]:
def build_full_text(example):
    return (
        f"Title: {example['title']}\n"
        f"Ingredients: {example['ingredients']}\n"
        f"Instructions: {example['directions']}"
    )

def build_prompt(example):
    return (
        f"Title: {example['title']}\n"
        f"Ingredients: {example['ingredients']}\n"
        "Instructions:"
    )

print(build_full_text(train_df.iloc[0]))

Title: No-Bake Chocolate Raspberry Cheesecake
Ingredients: 2 c. graham cracker crumbs 1/2 c. melted butter 1 envelope Knox gelatine 1/2 c. cold water 3 (8 oz.) softened cream cheese 1 1/4 c. sugar 1 (5 oz.) can evaporated milk 1 tsp. lemon juice 1 tsp. vanilla 2 c. heavy whipping cream 3 oz. chocolate chips 1/4 c. seedless raspberry preserves
Instructions: Combine graham cracker crumbs and melted butter. Press into spring-form pan, halfway up the sides. Chill.


## 8. Load tokenizer and tokenize datasets

In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 has no native padding token.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

def tokenize_batch(batch):
    texts = [
        (
            f"Title: {title}\n"
            f"Ingredients: {ingredients}\n"
            f"Instructions: {instructions}{tokenizer.eos_token}"
        )
        for title, ingredients, instructions in zip(
            batch["title"],
            batch["ingredients"],
            batch["directions"],
        )
    ]

    encoded = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    # Ignore padding tokens when calculating training loss.
    encoded["labels"] = [
        [
            token_id if attention == 1 else -100
            for token_id, attention in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(
            encoded["input_ids"], encoded["attention_mask"]
        )
    ]
    return encoded

tokenized = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing recipes",
)

tokenized

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizing recipes:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing recipes:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing recipes:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

## 9. Inspect truncation

A 256-token limit is chosen for Colab feasibility. This cell reports how often recipes exceed the limit before truncation.

In [10]:
def token_length(example):
    text = build_full_text(example)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

length_sample = train_df.sample(
    n=min(1000, len(train_df)), random_state=SEED
).apply(token_length, axis=1)

print(length_sample.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(
    f"Estimated fraction above {MAX_LENGTH} tokens:",
    f"{(length_sample > MAX_LENGTH).mean():.2%}",
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1240 > 1024). Running this sequence through the model will result in indexing errors


count    1000.000000
mean      193.944000
std       127.310435
min        38.000000
50%       157.000000
75%       241.250000
90%       348.400000
95%       437.200000
99%       653.200000
max      1240.000000
dtype: float64
Estimated fraction above 256 tokens: 22.10%


## 10. Load GPT-2 Small

In [17]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # Recommended while training.

print("Parameters:", f"{model.num_parameters():,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Parameters: 124,439,808


## 11. Configure Hugging Face Trainer

Checkpoints are saved directly to Google Drive. The code detects whether the installed Transformers version uses `eval_strategy` or the older `evaluation_strategy` argument.

In [18]:
import inspect
from transformers import TrainingArguments, Trainer, DefaultDataCollator

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    fp16=torch.cuda.is_available(),

    # Evaluate and save every 500 training steps
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=50,

    # Restore the checkpoint with the lowest validation loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    warmup_ratio=0.05,
    weight_decay=0.01,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

# Handle different Transformers versions
training_signature = inspect.signature(TrainingArguments.__init__)

if "eval_strategy" in training_signature.parameters:
    training_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in training_signature.parameters:
    training_kwargs["evaluation_strategy"] = "steps"
else:
    raise RuntimeError(
        "Your TrainingArguments version supports neither "
        "'eval_strategy' nor 'evaluation_strategy'."
    )

training_kwargs["eval_steps"] = 500

# Remove any arguments unsupported by the installed version
supported_training_kwargs = {
    name: value
    for name, value in training_kwargs.items()
    if name in training_signature.parameters
}

removed_arguments = set(training_kwargs) - set(supported_training_kwargs)

if removed_arguments:
    print("Unsupported arguments removed:", sorted(removed_arguments))

training_args = TrainingArguments(**supported_training_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DefaultDataCollator(),
)

# Newer versions use processing_class; older versions use tokenizer
trainer_signature = inspect.signature(Trainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_us

## 12. Train or resume from a checkpoint

Leave `RESUME_IF_AVAILABLE = True`. If Colab disconnects, rerunning the notebook will continue from the newest checkpoint.

In [13]:
RESUME_IF_AVAILABLE = True

checkpoint_paths = glob.glob(str(OUTPUT_DIR / "checkpoint-*"))
checkpoint_paths = sorted(
    checkpoint_paths,
    key=lambda path: int(Path(path).name.split("-")[-1]),
)

resume_checkpoint = checkpoint_paths[-1] if checkpoint_paths else None

if resume_checkpoint and RESUME_IF_AVAILABLE:
    print("Resuming from:", resume_checkpoint)
else:
    print("Starting a new training run.")
    resume_checkpoint = None

start_time = time.time()
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
elapsed_hours = (time.time() - start_time) / 3600

print(f"Training time: {elapsed_hours:.2f} hours")
train_result

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting a new training run.


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,2.409004,2.301495
1000,2.335027,2.222609
1500,2.237515,2.177203
2000,2.194702,2.152443
2500,2.179157,2.144190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training time: 0.42 hours


TrainOutput(global_step=2500, training_loss=2.31214387512207, metrics={'train_runtime': 1526.3207, 'train_samples_per_second': 26.207, 'train_steps_per_second': 1.638, 'total_flos': 5225840640000000.0, 'train_loss': 2.31214387512207, 'epoch': 2.0})

## 13. Save the final model, tokenizer, and training metrics

In [9]:
FINAL_MODEL_DIR = OUTPUT_DIR / "best_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

train_metrics = dict(train_result.metrics)
train_metrics["elapsed_hours_measured"] = elapsed_hours

suffix = "pilot" if PILOT_MODE else "final"

with open(
    RESULTS_DIR / f"training_metrics_{suffix}.json",
    "w"
) as f:
    json.dump(train_metrics, f, indent=2)

print("Saved model to:", FINAL_MODEL_DIR)
print("Saved metrics to:", RESULTS_DIR / f"training_metrics_{suffix}.json")

NameError: name 'trainer' is not defined

## 14. Evaluate validation loss and perplexity

In [15]:
# Evaluate on validation set
eval_metrics = trainer.evaluate()

eval_loss = float(eval_metrics["eval_loss"])

try:
    perplexity = math.exp(eval_loss)
except OverflowError:
    perplexity = float("inf")

eval_metrics["perplexity"] = perplexity

suffix = "pilot" if PILOT_MODE else "final"

with open(
    RESULTS_DIR / f"validation_metrics_{suffix}.json",
    "w"
) as f:
    json.dump(eval_metrics, f, indent=2)

print("Validation metrics")
print(eval_metrics)

Training Loss,Validation Loss,Step
2.179157,2.144190,2500


Validation metrics
{'eval_loss': 2.1441895961761475, 'perplexity': 8.53512153866571}


## 15. Quick generation sanity check

This verifies that the fine-tuned model can generate instructions from a title and ingredients.

In [16]:
model.config.use_cache = True
model.eval()

def generate_one(model, prompt, **generation_kwargs):
    device = next(model.parameters()).device
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=120,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **generation_kwargs,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

example = test_df.iloc[0].to_dict()
prompt = build_prompt(example)

print("PROMPT\n", prompt)
print("\nREFERENCE\n", example["directions"])
print("\nGENERATED\n", generate_one(model, prompt, do_sample=False))

PROMPT
 Title: Chocolate Roulage with Bourbon Cream
Ingredients: Softened unsalted butter, for greasing baking sheet 5 egg yolks 1 cup granulated sugar 3 ounces bittersweet chocolate, chopped 3 ounces semisweet chocolate, chopped 1 tablespoon espresso or strong black coffee 2 teaspoons vanilla extract 5 egg whites 2 cups heavy cream 2 teaspoons confectioners' sugar 1/2 cup bourbon Seeds from 1/2 vanilla bean 1 cup cocoa powder, plus more for dusting Fresh berries, for garnish
Instructions:

REFERENCE
 Preheat the oven to 325 degrees F. Butter a rimmed jelly roll/baking sheet and line with parchment paper. Butter the top of the paper as well, especially the corners. In a stand mixer with the whisk attachment, beat the egg yolks and granulated sugar on medium speed until fluffy and the sugar is not gritty, 10 to12 minutes. While the eggs are beating, put the chocolate in a bowl and melt over a double boiler. When the chocolate is melted, remove from the heat and let it cool a bit. Sit th

# Model and decoding comparison

The following sections generate outputs for fixed test prompts and save each experiment immediately.

## 16. Select a fixed generation subset

In [19]:
generation_df = test_df.sample(
    n=min(GENERATION_SAMPLE_SIZE, len(test_df)),
    random_state=SEED,
).reset_index(drop=True)

generation_df.insert(0, "recipe_id", range(len(generation_df)))
generation_subset_path = DATA_DIR / f"generation_subset_{RUN_SUFFIX}.csv"
generation_df.to_csv(generation_subset_path, index=False)

print("Generation prompts:", len(generation_df))
print("Saved subset to:", generation_subset_path)
generation_df.head()

Generation prompts: 500
Saved subset to: /content/drive/Shareddrives/RecipeGPT/data/generation_subset_final.csv


,recipe_id,title,ingredients,directions
0,0,Ham Balls,"2 c. ham, cubed 3 c. Bisquick 1 c. Cheddar che...",Combine all ingredients. Roll into nickel size...
1,1,Fisherman'S Wharf-Style Clam Chowder In Bread ...,2 tablespoons diced salt pork or 1 slice bacon...,Place salt pork or bacon in a pot and cook 2 m...
2,2,Brussels Sprouts With Cornbread Croutons,"2 lbs Brussels sprouts, trimmed and halved 1 1...",Preheat oven to 425°F. Toss together first 4 i...
3,3,Mexican Dumplings,1 lb. hamburger 1/2 small tomato (optional) 1 ...,Brown onion and garlic in oil. Add hamburger a...
4,4,Southwestern Marinade for Grilling,"1 limes, zest of or 1 lime, juice of 2 tablesp...",combine all ingredients in a small bowl and st...


## 17. Batch generation helper

Batching makes generation substantially faster than processing one recipe at a time.

In [20]:
def generate_dataset(
    model,
    frame,
    method_name,
    generation_kwargs,
    batch_size=8,
):
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device

    rows = []
    set_seed(SEED)
    tokenizer.padding_side = "left"

    try:
        for start in range(0, len(frame), batch_size):
            batch = frame.iloc[start:start + batch_size]
            prompts = [build_prompt(row) for _, row in batch.iterrows()]

            encoded = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)

            input_length = encoded["input_ids"].shape[1]

            with torch.no_grad():
                generated_ids = model.generate(
                    **encoded,
                    max_new_tokens=160,
                    no_repeat_ngram_size=3,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    **generation_kwargs,
                )

            completion_ids = generated_ids[:, input_length:]
            predictions = tokenizer.batch_decode(
                completion_ids,
                skip_special_tokens=True,
            )

            for (_, record), prediction in zip(batch.iterrows(), predictions):
                rows.append({
                    "recipe_id": int(record["recipe_id"]),
                    "title": record["title"],
                    "ingredients": record["ingredients"],
                    "reference": record["directions"],
                    "prediction": prediction.strip(),
                    "method": method_name,
                })
    finally:
        tokenizer.padding_side = "right"

    result = pd.DataFrame(rows)
    output_path = RESULTS_DIR / f"{method_name}_{RUN_SUFFIX}.csv"
    result.to_csv(output_path, index=False)
    print("Saved:", output_path)
    return result

## 18. Generate fine-tuned outputs using five decoding strategies

Run these one at a time if GPU time is limited. Each completed result is saved to Drive.

In [19]:
decoding_configs = {
    "finetuned_greedy": {
        "do_sample": False,
        "num_beams": 1,
    },
    "finetuned_beam": {
        "do_sample": False,
        "num_beams": 4,
        "early_stopping": True,
    },
    "finetuned_top_k": {
        "do_sample": True,
        "top_k": 50,
        "top_p": 1.0,
        "temperature": 1.0,
    },
    "finetuned_top_p": {
        "do_sample": True,
        "top_k": 0,
        "top_p": 0.90,
        "temperature": 1.0,
    },
    "finetuned_temperature": {
        "do_sample": True,
        "top_k": 0,
        "top_p": 1.0,
        "temperature": 0.7,
    },
}

# Ensure the fine-tuned model is on the active device.
model.to("cuda" if torch.cuda.is_available() else "cpu")
finetuned_results = {}

for method_name, config in decoding_configs.items():
    result_path = RESULTS_DIR / f"{method_name}_{RUN_SUFFIX}.csv"

    if result_path.exists():
        print("Loading existing result:", result_path)
        finetuned_results[method_name] = pd.read_csv(result_path)
    else:
        finetuned_results[method_name] = generate_dataset(
            model=model,
            frame=generation_df,
            method_name=method_name,
            generation_kwargs=config,
            batch_size=8,
        )

Saved: /content/drive/Shareddrives/RecipeGPT/results/finetuned_greedy_final.csv
Saved: /content/drive/Shareddrives/RecipeGPT/results/finetuned_beam_final.csv
Saved: /content/drive/Shareddrives/RecipeGPT/results/finetuned_top_k_final.csv
Saved: /content/drive/Shareddrives/RecipeGPT/results/finetuned_top_p_final.csv
Saved: /content/drive/Shareddrives/RecipeGPT/results/finetuned_temperature_final.csv


## 19. Generate pretrained GPT-2 baseline

The baseline uses greedy decoding on the exact same prompts. This isolates the effect of domain fine-tuning.

In [20]:
# Release GPU memory before loading the original pretrained model.
model.to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
pretrained_model.config.pad_token_id = tokenizer.pad_token_id
pretrained_model.to("cuda" if torch.cuda.is_available() else "cpu")

baseline_path = RESULTS_DIR / f"pretrained_greedy_{RUN_SUFFIX}.csv"

if baseline_path.exists():
    pretrained_result = pd.read_csv(baseline_path)
    print("Loaded existing baseline:", baseline_path)
else:
    pretrained_result = generate_dataset(
        model=pretrained_model,
        frame=generation_df,
        method_name="pretrained_greedy",
        generation_kwargs={"do_sample": False, "num_beams": 1},
        batch_size=8,
    )

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Saved: /content/drive/Shareddrives/RecipeGPT/results/pretrained_greedy_final.csv


## 20. Automatic evaluation functions

In [21]:
import evaluate

sacrebleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def distinct_n(texts, n):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        tokens = str(text).lower().split()
        ngrams = [
            tuple(tokens[i:i+n])
            for i in range(max(0, len(tokens) - n + 1))
        ]
        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    return len(unique_ngrams) / total_ngrams if total_ngrams else 0.0

def evaluate_predictions(result_df):
    predictions = result_df["prediction"].fillna("").astype(str).tolist()
    references = result_df["reference"].fillna("").astype(str).tolist()

    bleu = sacrebleu_metric.compute(
        predictions=predictions,
        references=[[reference] for reference in references],
    )

    rouge = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )

    bert = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en",
        batch_size=16,
    )

    lengths = [len(text.split()) for text in predictions]

    return {
        "BLEU": float(bleu["score"]),
        "ROUGE-L": float(rouge["rougeL"]),
        "BERTScore-F1": float(np.mean(bert["f1"])),
        "Distinct-1": float(distinct_n(predictions, 1)),
        "Distinct-2": float(distinct_n(predictions, 2)),
        "Average-length": float(np.mean(lengths)),
        "Empty-output-rate": float(
            np.mean([len(text.strip()) == 0 for text in predictions])
        ),
    }

## 21. Evaluate all saved outputs

BERTScore may take longer than BLEU and ROUGE. Results are saved after each method.

In [22]:
result_files = sorted(RESULTS_DIR.glob(f"*_{RUN_SUFFIX}.csv"))
result_files = [
    path for path in result_files
    if path.name.startswith(("finetuned_", "pretrained_"))
]

if not result_files:
    raise FileNotFoundError(
        "No generation result files were found. Run Sections 18 and 19 first."
    )

all_metrics = []

for path in result_files:
    print("Evaluating:", path.name)
    result_df = pd.read_csv(path)
    metrics = evaluate_predictions(result_df)
    metrics["method"] = path.stem.replace(f"_{RUN_SUFFIX}", "")
    all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics).set_index("method").sort_index()
metrics_output_path = RESULTS_DIR / f"generation_metrics_{RUN_SUFFIX}.csv"
metrics_df.to_csv(metrics_output_path)

print("Saved metrics to:", metrics_output_path)
metrics_df

Evaluating: finetuned_beam_final.csv


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating: finetuned_greedy_final.csv
Evaluating: finetuned_temperature_final.csv
Evaluating: finetuned_top_k_final.csv
Evaluating: finetuned_top_p_final.csv
Evaluating: pretrained_greedy_final.csv
Saved metrics to: /content/drive/Shareddrives/RecipeGPT/results/generation_metrics_final.csv


,BLEU,ROUGE-L,BERTScore-F1,Distinct-1,Distinct-2,Average-length,Empty-output-rate
method,,,,,,,
finetuned_beam,5.227434,0.214456,0.863537,0.053571,0.202339,47.862,0.0
finetuned_greedy,4.468649,0.214964,0.868235,0.066926,0.242536,42.734,0.0
finetuned_temperature,3.924066,0.194124,0.862614,0.089360,0.381274,51.410,0.0
finetuned_top_k,3.302478,0.175895,0.856637,0.103051,0.480776,61.756,0.0
finetuned_top_p,3.600608,0.183204,0.858050,0.100942,0.447455,60.054,0.0
pretrained_greedy,3.995123,0.162055,0.845693,0.045959,0.177879,116.888,0.0


## 22. Reload the fine-tuned model later (optional)

**<mark>Use this cell in a new session when you only need generation or evaluation.<mark>**

In [10]:
# Run this cell only in a new session when you want generation/evaluation
# without retraining. Remove the leading comments when needed.

# tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# model = AutoModelForCausalLM.from_pretrained(str(FINAL_MODEL_DIR))
# model.config.pad_token_id = tokenizer.pad_token_id
# model.to("cuda" if torch.cuda.is_available() else "cpu")
# model.eval()

# Optional stretch goal: LoRA

Run this only after the full fine-tuning and decoding experiments are complete.

## 23. Initialize a GPT-2 LoRA model

In [11]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [22]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

lora_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
lora_base_model.config.pad_token_id = tokenizer.pad_token_id
lora_base_model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    bias="none",
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()
lora_trainable_params, lora_total_params = lora_model.get_nb_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 24. Train LoRA with the same data split

In [26]:
LORA_OUTPUT_DIR = MODEL_DIR.parent / f"gpt2_lora_{RUN_SUFFIX}"
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

lora_kwargs = dict(supported_training_kwargs)
lora_kwargs["output_dir"] = str(LORA_OUTPUT_DIR)
lora_training_args = TrainingArguments(**lora_kwargs)

lora_trainer_kwargs = dict(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DefaultDataCollator(),
)

if "processing_class" in trainer_signature.parameters:
    lora_trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    lora_trainer_kwargs["tokenizer"] = tokenizer

lora_trainer = Trainer(**lora_trainer_kwargs)

lora_start_time = time.time()
lora_trainer.train()
lora_elapsed_hours = (time.time() - lora_start_time) / 3600

lora_trainer.save_model(str(LORA_OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR / "adapter"))

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss,Validation Loss
500,2.724832,2.571743
1000,2.708511,2.541588
1500,2.674478,2.529858
2000,2.664518,2.522488
2500,2.664678,2.520511


('/content/drive/Shareddrives/RecipeGPT/models/gpt2_lora_final/adapter/tokenizer_config.json',
 '/content/drive/Shareddrives/RecipeGPT/models/gpt2_lora_final/adapter/tokenizer.json')

## 24.1 Evaluation Lora vs full fine tuned gpt

In [27]:
# Same fixed prompts used for the finetuned/pretrained comparison.
lora_generation_path = RESULTS_DIR / f"lora_greedy_{RUN_SUFFIX}.csv"
if lora_generation_path.exists():
    print("Loading existing LoRA generation result:", lora_generation_path)
    lora_result = pd.read_csv(lora_generation_path)
else:
    lora_result = generate_dataset(
        model=lora_model,
        frame=generation_df,
        method_name="lora_greedy",
        generation_kwargs={"do_sample": False, "num_beams": 1},
        batch_size=8,
    )

lora_eval_metrics = lora_trainer.evaluate()
lora_eval_loss = float(lora_eval_metrics["eval_loss"])

try:
    lora_perplexity = math.exp(lora_eval_loss)
except OverflowError:
    lora_perplexity = float("inf")

lora_eval_metrics["perplexity"] = lora_perplexity

with open(RESULTS_DIR / f"lora_validation_metrics_{RUN_SUFFIX}.json", "w") as f:
    json.dump(lora_eval_metrics, f, indent=2)

# Generation-quality metrics — reuses the same scorer as every other method.
lora_generation_metrics = evaluate_predictions(lora_result)
lora_generation_metrics["method"] = "lora_greedy"

# Pull the matching full-fine-tune numbers out of what's already been saved,
# so the comparison table doesn't require re-running full-FT eval.
full_ft_metrics_path = RESULTS_DIR / f"generation_metrics_{RUN_SUFFIX}.csv"
full_ft_metrics_df = pd.read_csv(full_ft_metrics_path, index_col="method")

full_ft_training_metrics_path = RESULTS_DIR / f"training_metrics_{RUN_SUFFIX}.json"
with open(full_ft_training_metrics_path) as f:
    full_ft_training_metrics = json.load(f)

full_ft_validation_metrics_path = RESULTS_DIR / f"validation_metrics_{RUN_SUFFIX}.json"
with open(full_ft_validation_metrics_path) as f:
    full_ft_validation_metrics = json.load(f)

# Trainable-parameter count for the full fine-tune, for a fair side-by-side
# (every param is trainable in a full fine-tune, unlike LoRA).
full_ft_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
full_ft_total_params = sum(p.numel() for p in model.parameters())

comparison_df = pd.DataFrame([
    {
        "method": "full_finetune",
        "trainable_params": full_ft_trainable_params,
        "total_params": full_ft_total_params,
        "trainable_pct": 100 * full_ft_trainable_params / full_ft_total_params,
        "training_hours": full_ft_training_metrics.get("elapsed_hours_measured"),
        "perplexity": full_ft_validation_metrics.get("perplexity"),
        "BLEU": full_ft_metrics_df.loc["finetuned_greedy", "BLEU"],
        "ROUGE-L": full_ft_metrics_df.loc["finetuned_greedy", "ROUGE-L"],
        "BERTScore-F1": full_ft_metrics_df.loc["finetuned_greedy", "BERTScore-F1"],
    },
    {
        "method": "lora",
        "trainable_params": lora_trainable_params,
        "total_params": lora_total_params,
        "trainable_pct": 100 * lora_trainable_params / lora_total_params,
        "training_hours": lora_elapsed_hours,
        "perplexity": lora_perplexity,
        "BLEU": lora_generation_metrics["BLEU"],
        "ROUGE-L": lora_generation_metrics["ROUGE-L"],
        "BERTScore-F1": lora_generation_metrics["BERTScore-F1"],
    },
]).set_index("method")

comparison_path = RESULTS_DIR / f"lora_vs_finetune_comparison_{RUN_SUFFIX}.csv"
comparison_df.to_csv(comparison_path)

print("Saved comparison to:", comparison_path)
comparison_df


Saved: /content/drive/Shareddrives/RecipeGPT/results/lora_greedy_final.csv


Training Loss,Validation Loss,Step
2.664678,2.520511,2500


Saved comparison to: /content/drive/Shareddrives/RecipeGPT/results/lora_vs_finetune_comparison_final.csv


,trainable_params,total_params,trainable_pct,training_hours,perplexity,BLEU,ROUGE-L,BERTScore-F1
method,,,,,,,,
full_finetune,124439808,124439808,100.000000,0.424232,8.535122,4.468649,0.214964,0.868235
lora,294912,124734720,0.236431,0.322059,12.434948,4.070354,0.197117,0.862776


## *24.2 Lora extension (if extra time is available)*
*Investigate one aspect of LoRA, such as:*

*   *Different LoRA ranks (e.g., r = 4, 8, 16)*
*   *Different alpha values*
*   *Different training dataset sizes*
*   *Different numbers of training epochs*



In [23]:
# Investigates whether generation quality / perplexity keeps improving with rank, or plateaus

RANKS_TO_SWEEP = [4, 8, 16]
rank_sweep_rows = []

for r in RANKS_TO_SWEEP:
    print(f"\n{'='*60}\nRank sweep: r={r}\n{'='*60}")

    rank_output_dir = MODEL_DIR.parent / f"gpt2_lora_{RUN_SUFFIX}_r{r}"
    adapter_dir = rank_output_dir / "adapter"

    # --- Load or train this rank's adapter ---
    if adapter_dir.exists():
        print("Loading existing adapter:", adapter_dir)
        rank_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        rank_base_model.config.pad_token_id = tokenizer.pad_token_id
        rank_model = PeftModel.from_pretrained(rank_base_model, str(adapter_dir))
        rank_model.to("cuda" if torch.cuda.is_available() else "cpu")

        # Reload previously captured training-time metrics if present.
        rank_metrics_path = rank_output_dir / "rank_run_metrics.json"
        try:
            with open(rank_metrics_path) as f:
                saved_run_metrics = json.load(f)
            rank_elapsed_hours = saved_run_metrics["training_hours"]
        except (FileNotFoundError, json.JSONDecodeError, KeyError):
            print(f"Warning: missing/corrupt metrics file for r={r}; training_hours unavailable.")
            rank_elapsed_hours = None
    else:
        rank_output_dir.mkdir(parents=True, exist_ok=True)

        rank_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        rank_base_model.config.pad_token_id = tokenizer.pad_token_id
        rank_base_model.config.use_cache = False

        rank_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=r,
            lora_alpha=16,          # alpha fixed at 16, matching the r=8 config used in previous cells.
            lora_dropout=0.05,
            target_modules=["c_attn"],
            bias="none",
        )
        rank_model = get_peft_model(rank_base_model, rank_config)
        rank_model.print_trainable_parameters()

        rank_kwargs = dict(supported_training_kwargs)
        rank_kwargs["output_dir"] = str(rank_output_dir)
        rank_training_args = TrainingArguments(**rank_kwargs)

        rank_trainer_kwargs = dict(
            model=rank_model,
            args=rank_training_args,
            train_dataset=tokenized["train"],
            eval_dataset=tokenized["validation"],
            data_collator=DefaultDataCollator(),
        )
        if "processing_class" in trainer_signature.parameters:
            rank_trainer_kwargs["processing_class"] = tokenizer
        elif "tokenizer" in trainer_signature.parameters:
            rank_trainer_kwargs["tokenizer"] = tokenizer

        rank_trainer = Trainer(**rank_trainer_kwargs)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        rank_start_time = time.time()
        rank_trainer.train()
        rank_elapsed_hours = (time.time() - rank_start_time) / 3600

        rank_trainer.save_model(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))

        with open(rank_output_dir / "rank_run_metrics.json", "w") as f:
            json.dump(
                {"training_hours": rank_elapsed_hours},
                f,
                indent=2,
            )

        # Free the trainer's held references before eval reuses the model.
        rank_model = rank_trainer.model

    rank_trainable_params, rank_total_params = rank_model.get_nb_trainable_parameters()

    # --- Validation perplexity ---
    # Build a lightweight Trainer for eval only (works whether we just
    # trained or reloaded from disk, since reloaded adapters have no trainer).
    eval_only_trainer = Trainer(
        model=rank_model,
        args=TrainingArguments(
            output_dir=str(rank_output_dir / "eval_scratch"),
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            report_to="none",
        ),
        eval_dataset=tokenized["validation"],
        data_collator=DefaultDataCollator(),
    )
    rank_eval_metrics = eval_only_trainer.evaluate()
    rank_eval_loss = float(rank_eval_metrics["eval_loss"])

    try:
        rank_perplexity = math.exp(rank_eval_loss)
    except OverflowError:
        rank_perplexity = float("inf")

    # --- Generation quality on the same fixed prompts ---
    rank_generation_path = RESULTS_DIR / f"lora_r{r}_greedy_{RUN_SUFFIX}.csv"

    if rank_generation_path.exists():
        print("Loading existing generation result:", rank_generation_path)
        rank_generation_result = pd.read_csv(rank_generation_path)
    else:
        rank_generation_result = generate_dataset(
            model=rank_model,
            frame=generation_df,
            method_name=f"lora_r{r}_greedy",
            generation_kwargs={"do_sample": False, "num_beams": 1},
            batch_size=8,
        )

    rank_generation_metrics = evaluate_predictions(rank_generation_result)

    rank_sweep_rows.append({
        "r": r,
        "trainable_params": rank_trainable_params,
        "total_params": rank_total_params,
        "trainable_pct": 100 * rank_trainable_params / rank_total_params,
        "training_hours": rank_elapsed_hours,
        "perplexity": rank_perplexity,
        "BLEU": rank_generation_metrics["BLEU"],
        "ROUGE-L": rank_generation_metrics["ROUGE-L"],
        "BERTScore-F1": rank_generation_metrics["BERTScore-F1"],
        "Distinct-1": rank_generation_metrics["Distinct-1"],
        "Distinct-2": rank_generation_metrics["Distinct-2"],
    })

    # Free GPU memory before the next rank.
    rank_model.to("cpu")
    del rank_model, rank_base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

rank_sweep_df = pd.DataFrame(rank_sweep_rows).set_index("r").sort_index()
rank_sweep_path = RESULTS_DIR / f"lora_rank_sweep_{RUN_SUFFIX}.csv"
rank_sweep_df.to_csv(rank_sweep_path)

print("\nSaved rank sweep to:", rank_sweep_path)
rank_sweep_df


Rank sweep: r=4


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1184


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

Loading existing generation result: /content/drive/Shareddrives/RecipeGPT/results/lora_r8_greedy_final.csv

Rank sweep: r=16
Loading existing adapter: /content/drive/Shareddrives/RecipeGPT/models/gpt2_lora_final_r16/adapter


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
No log,2.520660,0


Loading existing generation result: /content/drive/Shareddrives/RecipeGPT/results/lora_r16_greedy_final.csv

Saved rank sweep to: /content/drive/Shareddrives/RecipeGPT/results/lora_rank_sweep_final.csv


,trainable_params,total_params,trainable_pct,training_hours,perplexity,BLEU,ROUGE-L,BERTScore-F1,Distinct-1,Distinct-2
r,,,,,,,,,,
4,147456,124587264,0.118356,0.309665,12.457310,4.194907,0.196367,0.862715,0.054353,0.199661
8,0,124734720,0.000000,0.314424,12.444635,3.923836,0.196253,0.862629,0.055436,0.202954
16,0,125029632,0.000000,0.314280,12.436807,3.874607,0.196995,0.862396,0.054996,0.203861


## Completion checklist

The core project is complete when you have:

- A successful pilot run.
- A final GPT-2 Small model trained on 20,000 recipes.
- Saved checkpoints and validation perplexity.
- Pretrained and fine-tuned greedy-decoding outputs.
- Five fine-tuned decoding-strategy output files.
- One automatic-metrics comparison table.
- LoRA results only if time remains.